# All turbine features: one year

Run **02_parquet_setup/01_setup_yearly_parquets.ipynb** to prepare source data, then **03_preprocessing/01_preprocess_10min_parquets.ipynb** to align it. Set `YEAR`, `TURBINE_ID` and `WINDOW_START` here and click **Run All**. This notebook reads only `Data_Parquets_10min/<year>/Station_<ID>/data.parquet`; it never performs conversion. After source changes, rerun 10 and 13 first.

Every sensor feature has its own plotting cell. Use the fast redraw cell at the bottom for later date changes without loading files again. Month/year durations follow calendar lengths; the end is exclusive. Missing values remain gaps, and empty features are reported without stopping other plots. Solid red lines mark timestamps with recorded alarm codes, regardless of whether an end time is available. Simultaneous codes share one line. No duration shading is shown. Alarms are not necessarily related to the plotted sensor; events starting outside the exported year may be absent. Set `SHOW_ALARMS = False` to hide overlays. Units should be checked against the dataset documentation.

Alarm lines are coloured by code using `ALARM_COLORS` in settings. The legend lists codes present in the displayed window. Events at identical timestamps overlap; all their codes remain listed in the legend.

**Updated input:** Run notebook 02_parquet_setup/01_setup_yearly_parquets.ipynb, then **03_preprocessing/01_preprocess_10min_parquets.ipynb**. This notebook reads `Data_Parquets_10min/<year>/Station_<ID>/data.parquet`. Alarm lines are aligned to the interval containing the alarm start. Missing intervals stay blank.


In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

WORKSPACE_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                       if (p / 'notebooks' / '01_data_export/01_export_yearly_turbine_scada.ipynb').is_file()), None)
if WORKSPACE_ROOT is None:
    raise FileNotFoundError('Open this notebook from Iteration2 or its notebooks folder.')

import sys
if str(WORKSPACE_ROOT / "notebooks" / "shared") not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT / "notebooks" / "shared"))
from interval_parquet import load_aligned


In [2]:
# EDIT THESE SETTINGS.
YEAR = 2016
TURBINE_ID = '2304527'
WINDOW_START = f'{YEAR}-01-01'
DURATION = pd.DateOffset(years=1)
SHOW_ALARMS = True
PARQUET_ROOT = WORKSPACE_ROOT / "Data_Parquets_10min"
ALARM_CODES = {'102', '1005', '3130', '8000', '8210', '8230', '8234', '8235', '8236', '10105'}  # Yellow-highlighted codes only.

# Consistent colours across all feature plots and duration notebooks.
ALARM_COLORS = {'102': '#e41a1c', '1005': '#ff7f00', '3130': '#4daf4a', '8000': '#984ea3', '8210': '#a65628', '8230': '#f781bf', '8234': '#999999', '8235': '#17becf', '8236': '#bcbd22', '10105': '#377eb8'}


In [3]:
# Read the regular interval dataset made by notebook 03_preprocessing/01_preprocess_10min_parquets.ipynb; no conversion here.
PARQUET_FOLDER = PARQUET_ROOT / str(YEAR) / f'Station_{TURBINE_ID}'
path = PARQUET_FOLDER / 'data.parquet'
if not path.is_file():
    raise FileNotFoundError(f'Run 03_preprocessing/01_preprocess_10min_parquets.ipynb first. Missing: {path}')
stat = path.stat()
signature = (str(path.resolve()), stat.st_size, stat.st_mtime_ns)
key = (PARQUET_FOLDER, f'Station_{TURBINE_ID}')
if '_aligned_signature' not in globals() or _aligned_signature != signature:
    interval_data, year_alarms = load_aligned(PARQUET_ROOT, YEAR, TURBINE_ID, bin_alarm_times=True)
    FEATURES = [c for c in interval_data if c.startswith('wtc_')]
    year_data = interval_data[FEATURES].sort_index()
    _aligned_signature = signature
    _prepared_key = key
FEATURES = list(year_data.columns)
print(f'10-minute dataset ready: {len(year_data):,} intervals; {len(year_alarms):,} recorded events.')


10-minute dataset ready: 52,704 intervals; 151,599 recorded events.


In [4]:
def select_window(window_start, duration, show_alarms=True, alarm_codes=None):
    global start, end, window_data, merged_intervals, markers, alarm_markers_by_code
    if '_prepared_key' not in globals():
        raise RuntimeError('Run the Parquet loading cell first.')
    if _prepared_key != (PARQUET_ROOT / str(YEAR) / f'Station_{TURBINE_ID}', f'Station_{TURBINE_ID}'):
        raise RuntimeError('The year or turbine changed. Rerun the Parquet loading cell first.')
    new_start = pd.Timestamp(window_start)
    length = duration if isinstance(duration, pd.DateOffset) else pd.Timedelta(duration)
    new_end = new_start + length
    if pd.isna(new_start) or pd.isna(new_end) or new_end <= new_start:
        raise ValueError('Choose a valid start date and positive duration.')
    # Binary search on the sorted timestamp index; the end is exclusive.
    left = year_data.index.searchsorted(new_start, side='left')
    right = year_data.index.searchsorted(new_end, side='left')
    selected = year_data.iloc[left:right].reset_index()
    # One line per recorded alarm timestamp with a nonblank selected code.
    new_markers = []
    new_markers_by_code = {}
    if show_alarms:
        alarms = year_alarms
        codes = alarms['Alarmcode'].astype('string').str.strip()
        alarms = alarms.loc[codes.notna() & codes.ne('')]
        if alarm_codes is not None:
            alarms = alarms.loc[alarms['Alarmcode'].astype(str).isin({str(c) for c in alarm_codes})]
        visible = alarms.loc[alarms['TimeOn'].ge(new_start) & alarms['TimeOn'].lt(new_end)].copy()
        visible['Alarmcode'] = visible['Alarmcode'].astype(str).str.strip()
        new_markers_by_code = {code: sorted(set(group['TimeOn']))
                               for code, group in visible.groupby('Alarmcode')}
        new_markers = sorted(set(visible['TimeOn']))
    globals()['_plotted_alarm_filters'] = {'enabled': show_alarms, 'included': None if alarm_codes is None else set(map(str, alarm_codes)), 'excluded': set()}
    start, end, window_data = new_start, new_end, selected
    merged_intervals, markers = [], new_markers
    alarm_markers_by_code = new_markers_by_code
    print(f'Window: {start} to {end} (exclusive); {len(window_data):,} sensor rows.')
    if window_data.empty:
        print('No readings in this window; choose dates in the loaded Parquet files.')

select_window(WINDOW_START, DURATION, SHOW_ALARMS, ALARM_CODES)


Window: 2016-01-01 00:00:00 to 2017-01-01 00:00:00 (exclusive); 52,704 sensor rows.


In [5]:
def plot_feature(feature):
    if feature not in window_data.columns:
        print(f'{feature}: column unavailable in this workbook.')
        return
    values = window_data[feature]
    fig, ax = plt.subplots(figsize=(15, 4))
    line, = ax.plot(window_data['TimeStamp'], values, color='tab:blue', linewidth=0.6, label=feature)
    handles = [line]
    for code, times in sorted(alarm_markers_by_code.items()):
        colour = ALARM_COLORS.get(code, '#555555')
        # One collection per code is faster than a separate artist per event.
        ax.vlines(mdates.date2num(times), 0, 1, transform=ax.get_xaxis_transform(),
                  color=colour, alpha=0.65, linewidth=0.8)
        handles.append(Line2D([0], [0], color=colour, linewidth=1.5,
                              label=f'Alarm {code}'))
    if not values.notna().any():
        ax.text(0.5, 0.5, 'No numeric readings for this feature in this window',
                transform=ax.transAxes, ha='center')
    ax.set(title=f'Turbine {TURBINE_ID} | {feature} | {start:%Y-%m-%d} to {end:%Y-%m-%d}',
           xlabel='Time', ylabel=feature, xlim=(start, end))
    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.grid(True, alpha=0.25)
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.close(fig)  # Release figures when running every feature cell.


### wtc_GearPres_mean


In [ ]:
plot_feature('wtc_GearPres_mean')


### wtc_HydPress_mean


In [ ]:
plot_feature('wtc_HydPress_mean')


### wtc_HubPresA_mean


In [ ]:
plot_feature('wtc_HubPresA_mean')


### wtc_HubPresB_mean


In [ ]:
plot_feature('wtc_HubPresB_mean')


### wtc_HubPresC_mean


In [ ]:
plot_feature('wtc_HubPresC_mean')


### wtc_HSGenTmp_mean


In [ ]:
plot_feature('wtc_HSGenTmp_mean')


### wtc_HSRotTmp_mean


In [ ]:
plot_feature('wtc_HSRotTmp_mean')


### wtc_IMSGenTm_mean


In [ ]:
plot_feature('wtc_IMSGenTm_mean')


### wtc_IMSRotTm_mean


In [ ]:
plot_feature('wtc_IMSRotTm_mean')


### wtc_GeOilTmp_mean


In [ ]:
plot_feature('wtc_GeOilTmp_mean')


### wtc_GenBeGTm_mean


In [ ]:
plot_feature('wtc_GenBeGTm_mean')


### wtc_GenBeRTm_mean


In [ ]:
plot_feature('wtc_GenBeRTm_mean')


### wtc_MainBTmp_mean


In [ ]:
plot_feature('wtc_MainBTmp_mean')


### wtc_Gen1U1Tm_mean


In [ ]:
plot_feature('wtc_Gen1U1Tm_mean')


### wtc_Gen1V1Tm_mean


In [ ]:
plot_feature('wtc_Gen1V1Tm_mean')


### wtc_Gen1W1Tm_mean


In [ ]:
plot_feature('wtc_Gen1W1Tm_mean')


### wtc_AmbieTmp_mean


In [ ]:
plot_feature('wtc_AmbieTmp_mean')


### wtc_NacelTmp_mean


In [ ]:
plot_feature('wtc_NacelTmp_mean')


### wtc_ConvWTmp_mean


In [ ]:
plot_feature('wtc_ConvWTmp_mean')


### wtc_InvUpTmp_mean


In [ ]:
plot_feature('wtc_InvUpTmp_mean')


### wtc_TraOilTF_mean


In [ ]:
plot_feature('wtc_TraOilTF_mean')


### wtc_TraRooTF_mean


In [ ]:
plot_feature('wtc_TraRooTF_mean')


### wtc_TrafoTMa_mean


In [ ]:
plot_feature('wtc_TrafoTMa_mean')


### wtc_HydOilTm_mean


In [ ]:
plot_feature('wtc_HydOilTm_mean')


### wtc_A1ExtTmp_mean


In [ ]:
plot_feature('wtc_A1ExtTmp_mean')


### wtc_A3LefTmp_mean


In [ ]:
plot_feature('wtc_A3LefTmp_mean')


### wtc_A3RigTmp_mean


In [ ]:
plot_feature('wtc_A3RigTmp_mean')


### wtc_A9IntTmp_mean


In [ ]:
plot_feature('wtc_A9IntTmp_mean')


### wtc_A21IntTm_mean


In [ ]:
plot_feature('wtc_A21IntTm_mean')


### wtc_DeltaTmp_mean


In [ ]:
plot_feature('wtc_DeltaTmp_mean')


### wtc_BrkTmpGn_mean


In [ ]:
plot_feature('wtc_BrkTmpGn_mean')


### wtc_GFilB1Tm_mean


In [ ]:
plot_feature('wtc_GFilB1Tm_mean')


### wtc_GFilB2Tm_mean


In [ ]:
plot_feature('wtc_GFilB2Tm_mean')


### wtc_GFilB3Tm_mean


In [ ]:
plot_feature('wtc_GFilB3Tm_mean')


### wtc_ReacUTmp_mean


In [ ]:
plot_feature('wtc_ReacUTmp_mean')


### wtc_ReacVTmp_mean


In [ ]:
plot_feature('wtc_ReacVTmp_mean')


### wtc_ReacWTmp_mean


In [ ]:
plot_feature('wtc_ReacWTmp_mean')


### wtc_PrWindSp_mean


In [ ]:
plot_feature('wtc_PrWindSp_mean')


### wtc_PrWindSp_stddev


In [ ]:
plot_feature('wtc_PrWindSp_stddev')


### wtc_GenRpm_mean


In [ ]:
plot_feature('wtc_GenRpm_mean')


### wtc_GenRpm_stddev


In [ ]:
plot_feature('wtc_GenRpm_stddev')


### wtc_MainSRpm_mean


In [ ]:
plot_feature('wtc_MainSRpm_mean')


### wtc_PowerRef_endvalue


In [ ]:
plot_feature('wtc_PowerRef_endvalue')


## Fast date change and redraw
Change the window here after setup; cached sensor data and alarms are reused.


In [ ]:
# Change these settings and rerun only this cell after setup.
VIEW_START = f'{YEAR}-01-01'
VIEW_DURATION = DURATION  # Or pd.DateOffset(days=1, weeks=0), months=1, years=1.
VIEW_FEATURES = None  # All features; or ['wtc_GearPres_mean', 'wtc_AmbieTmp_mean'].

select_window(VIEW_START, VIEW_DURATION, SHOW_ALARMS, ALARM_CODES)
for feature in (FEATURES if VIEW_FEATURES is None else VIEW_FEATURES):
    plot_feature(feature)


## Alarm trigger counts
Run after choosing the plotted period; counts recorded starts by code using the current plot filters. Repeated starts count separately; alarms starting before the window do not count as new triggers. Rerun after changing dates.


In [ ]:
# Run after plotting; rerun this cell whenever you change the window.
if not all(k in globals() for k in ('year_alarms', 'start', 'end')):
    raise RuntimeError('Run the data-loading and window-selection cells first.')
f = globals().get('_plotted_alarm_filters', {})
events = year_alarms.loc[year_alarms['TimeOn'].ge(start) & year_alarms['TimeOn'].lt(end)].copy()
events['Alarmcode'] = events['Alarmcode'].astype('string').str.strip()
events = events.loc[events['Alarmcode'].notna() & events['Alarmcode'].ne('')]
included = f.get('included', globals().get('ALARM_CODES'))
excluded = f.get('excluded', globals().get('EXCLUDED_ALARM_CODES', set()))
if included is not None:
    events = events.loc[events['Alarmcode'].isin({str(c).strip() for c in included})]
if excluded:
    events = events.loc[~events['Alarmcode'].isin({str(c).strip() for c in excluded})]
if not f.get('enabled', globals().get('SHOW_ALARMS', True)):
    events = events.iloc[:0]
alarm_trigger_counts = events.groupby('Alarmcode').size().rename('Trigger count').reset_index()
alarm_trigger_counts = alarm_trigger_counts.sort_values(['Trigger count', 'Alarmcode'], ascending=[False, True])
print(f'Alarm starts from {start} to {end} (end exclusive).')
print(f'Total: {len(events):,} triggers across {len(alarm_trigger_counts):,} codes.')
print('Counts follow the plotted alarm filters. Alarms starting before this window are not new triggers.')
from IPython.display import display
display(alarm_trigger_counts)
